<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/14_Deep_Learning/03_RNN_Temporal_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 14: Deep Learning Foundations
## Part 03: Recurrent Neural Networks (RNN) for Temporal Forecasting

### The Temporal Challenge
Environmental anomalies do not happen instantly. A severe ecosystem carbon emission spike is usually the result of compound stress building up over days or weeks—such as prolonged drought followed by an extreme heatwave.

Previous models (MLP and CNN) evaluate single, isolated snapshots. To forecast future anomalies, we must process sequential data. We will utilize Recurrent Neural Networks (RNNs)—specifically Long Short-Term Memory (LSTM) networks—which possess internal memory loops. This allows the network to remember the climate conditions from Tuesday when making a prediction for Friday.

### Objective
Construct a sequence-to-vector LSTM network capable of analyzing a trailing 7-day window of micrometeorological data to forecast the exact numerical value of the ecosystem stress index 24 hours into the future.

### Key Milestones
1. **Temporal Windowing:** Transform standard flat tabular data into 3D sequential tensors formatted as `(batch_size, time_steps, features)`.
2. **LSTM Architecture:** Construct a recurrent network that preserves gradients over time to capture long-term ecological dependencies.
3. **Continuous Forecasting:** Shift from binary classification (0 or 1) to continuous regression, optimizing for Mean Absolute Error (MAE) to predict precise future stress index values.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Verify environment
print(f"TensorFlow Version: {tf.__version__}")

# Reproducibility seed for sequence generation
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

TensorFlow Version: 2.21.0


### Part 02: Sliding Windows & 3D Sequence Tensors

An MLP processes a 1D vector `(features)`. A CNN processes a 2D spatial grid `(height, width, channels)`. An RNN requires a 3D tensor shaped as `(samples, time_steps, features)`.

To forecast the future, we cannot simply feed the network randomized rows of data. We must construct **sliding windows**. If our goal is to use the past 7 days of weather to predict day 8, we group rows 1 through 7 into a single sequence matrix, and set row 8's ecosystem stress index as the target. The window then slides forward by one day: rows 2 through 8 become the next sequence, targeting day 9.

Because time flows in one direction, we must split our Train, Validation, and Test sets chronologically. We cannot randomly shuffle the data, as that would cause "data leakage" by allowing the network to use future events to predict the past.

In [2]:
from sklearn.preprocessing import MinMaxScaler

# 1. Generate Continuous Temporal Sensor Data (Simulating 3,000 days)
days = np.arange(3000)
# Create compound weather signals with seasonal sine waves and daily noise
temperature = 25 + 10 * np.sin(days * 2 * np.pi / 365) + np.random.normal(0, 2, 3000)
soil_moisture = 40 + 15 * np.cos(days * 2 * np.pi / 365) + np.random.normal(0, 3, 3000)
radiation = 400 + 200 * np.sin(days * 2 * np.pi / 365) + np.random.normal(0, 50, 3000)

# The target continuous variable: Ecosystem Stress Index
# High stress occurs with high temp, high radiation, and low moisture
stress_index = (temperature * 1.5) + (radiation * 0.05) - (soil_moisture * 1.2) + np.random.normal(0, 2, 3000)

df_time = pd.DataFrame({
    'Temperature': temperature,
    'Soil_Moisture': soil_moisture,
    'Radiation': radiation,
    'Stress_Index': stress_index
})

# 2. Chronological Partitioning (70% Train, 15% Val, 15% Test)
train_size = int(len(df_time) * 0.70)
val_size = int(len(df_time) * 0.15)

train_df = df_time.iloc[:train_size]
val_df = df_time.iloc[train_size:train_size+val_size]
test_df = df_time.iloc[train_size+val_size:]

# 3. Fit scaling strictly on the training partition
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler() # We scale the target separately so we can inverse-transform predictions later

X_train_scaled = scaler_X.fit_transform(train_df[['Temperature', 'Soil_Moisture', 'Radiation']])
y_train_scaled = scaler_y.fit_transform(train_df[['Stress_Index']])

X_val_scaled = scaler_X.transform(val_df[['Temperature', 'Soil_Moisture', 'Radiation']])
y_val_scaled = scaler_y.transform(val_df[['Stress_Index']])

X_test_scaled = scaler_X.transform(test_df[['Temperature', 'Soil_Moisture', 'Radiation']])
y_test_scaled = scaler_y.transform(test_df[['Stress_Index']])

# 4. Function to generate 3D sliding windows
def create_sequences(X, y, time_steps=7):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

# Create 7-day lookback sequences
TIME_STEPS = 7
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, TIME_STEPS)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val_scaled, TIME_STEPS)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, TIME_STEPS)

print("✅ Temporal Windowing Complete.")
print("-" * 45)
print(f"3D Training Tensors: X: {X_train_seq.shape} | y: {y_train_seq.shape}")
print(f"3D Validation Tensors: X: {X_val_seq.shape}   | y: {y_val_seq.shape}")
print(f"3D Testing Tensors: X: {X_test_seq.shape}   | y: {y_test_seq.shape}")
print("-" * 45)

✅ Temporal Windowing Complete.
---------------------------------------------
3D Training Tensors: X: (2093, 7, 3) | y: (2093, 1)
3D Validation Tensors: X: (443, 7, 3)   | y: (443, 1)
3D Testing Tensors: X: (443, 7, 3)   | y: (443, 1)
---------------------------------------------


#Explanation:
The shape (2093, 7, 3) represents the exact 3-dimensional architecture required by a Recurrent Neural Network:

2093 (Samples): The total number of independent sequences (or "flashcards") the network will study during training.

7 (Time Steps): The historical lookback window. Every single sequence contains 7 consecutive days of data.

3 (Features): The environmental variables measured on each of those days (Temperature, Soil Moisture, and Radiation).

The target tensor y shaped as (2093, 1) means there are 2093 corresponding answers. For every 7-day sequence the network studies, there is exactly 1 target value it is trying to guess: the ecosystem stress index for day 8.

### Part 03: Architecting the Long Short-Term Memory (LSTM) Engine

Unlike a standard Multilayer Perceptron, an LSTM layer contains internal memory cells governed by "gates." As the LSTM steps through the 7-day sequence, these gates mathematically decide what information to keep (e.g., a prolonged heatwave) and what to forget (e.g., minor daily noise) before passing its final understanding to the Dense layers.

Furthermore, we are fundamentally shifting our objective. In previous models, we performed **Binary Classification** (0 or 1) using a `sigmoid` activation and evaluated using AUC. Now, we are performing **Continuous Regression**. The model must predict a precise numerical value.
* **Output Node:** Uses a `linear` activation function (no bounding).
* **Loss Function:** Mean Squared Error (MSE), which heavily penalizes predictions that are numerically far from the actual value.
* **Metric:** Mean Absolute Error (MAE), which tells us on average exactly how far off our predictions are.

In [3]:
# 1. Architect the LSTM Engine
lstm_model = keras.Sequential([
    layers.Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),

    # Recurrent Layer: Extracting temporal dependencies across the 7 days
    layers.LSTM(units=32, return_sequences=False),
    layers.Dropout(0.2),

    # Dense Feature Compression
    layers.Dense(units=16, activation='relu'),

    # Continuous Regression Output (No activation bounds)
    layers.Dense(units=1, activation='linear')
])

# 2. Compile for Regression
lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',  # Penalizes large numerical errors
    metrics=['mean_absolute_error']
)

# Display the architectural map
lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         4,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,153 (20.13 KB)

 Trainable params: 5,153 (20.13 KB)

 Non-trainable params: 0 (0.00 B)